# Audio Reverberation Analysis

Acoustic analysis of a 3-second audio recording to estimate reverberation parameters:
- **RT60**: Time for sound to decay by 60 dB
- **T20**: Time for sound to decay by 20 dB (extrapolated to 60 dB)
- **T30**: Time for sound to decay by 30 dB (extrapolated to 60 dB)
- **EDT**: Early Decay Time (from 0 dB to -10 dB)

The analysis uses the Schroeder integral to compute the decay envelope.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sounddevice as sd
from scipy import signal
from scipy.interpolate import interp1d
%matplotlib widget

In [ ]:
# Parameters
duration = 3.0  # seconds
fs = 44100  # sampling rate (Hz)
channels = 1  # mono recording

print("\nAvailable audio devices:")
print(sd.query_devices())
mic = 1
spk = 2
sd.default.device = (mic, spk)

In [ ]:

print("Recording audio for 3 seconds...")
print("Please make a clap/impulse sound to measure reverberation.")


# Record audio
audio_data = sd.rec(int(duration * fs), samplerate=fs, channels=channels, dtype=np.float32)
sd.wait()

# Convert to mono if needed
if audio_data.ndim > 1:
    audio = audio_data[:, 0]
else:
    audio = audio_data

# Normalize
audio = audio / np.max(np.abs(audio))

print(f"Recording completed. Shape: {audio.shape}, Duration: {len(audio)/fs:.3f} s")

In [ ]:
# Step 1: Identify the clap (loudest point in the signal)
clap_idx = np.argmax(np.abs(audio))
clap_idx+=100
clap_time = clap_idx / fs

print(f"Clap detected at: {clap_time:.4f} s (sample index: {clap_idx})")

# Limit analysis to after the clap
audio_after_clap = audio[clap_idx:]
time_after_clap = np.arange(len(audio_after_clap)) / fs

In [ ]:
# Step 2: Compute the Schroeder integral (energy decay envelope)
# The Schroeder integral is computed by backward integration of squared signal
squared_audio_after_clap = audio_after_clap ** 2

# Backward integration (integrate from end to beginning)
schroeder_integral = np.zeros_like(squared_audio_after_clap)
schroeder_integral[-1] = squared_audio_after_clap[-1]

for i in range(len(squared_audio_after_clap) - 2, -1, -1):
    schroeder_integral[i] = schroeder_integral[i + 1] + squared_audio_after_clap[i]

# Normalize the Schroeder integral
schroeder_integral = schroeder_integral / np.max(schroeder_integral)

# Convert to dB scale (0 dB at the peak)
# Avoid log(0) by adding a small epsilon
epsilon = 1e-10
envelope_db = 10 * np.log10(schroeder_integral + epsilon)

print(f"Envelope computed. Min dB: {np.min(envelope_db):.2f}, Max dB: {np.max(envelope_db):.2f}")
print(f"Time array length: {len(time_after_clap)}, Envelope length: {len(envelope_db)}")

In [ ]:
# Step 3: Extract reverberation metrics from the decay envelope

def find_decay_time(envelope_db, time_array, fs, start_db, end_db):
    """
    Find the time for the envelope to decay from start_db to end_db dB
    
    Parameters:
    - envelope_db: decay envelope in dB
    - time_array: corresponding time array
    - fs: sampling rate
    - start_db: starting dB level (typically 0)
    - end_db: ending dB level (e.g., -10, -20, -30, -60)
    
    Returns:
    - decay_time: time interval between the two levels
    - time_start: time at start_db level
    - time_end: time at end_db level
    - None if the envelope doesn't reach the required levels
    """
    # Find indices where envelope is above start_db and below end_db
    idx_start = np.where(envelope_db <= start_db)[0]
    idx_end = np.where(envelope_db <= end_db)[0]
    
    if len(idx_start) == 0 or len(idx_end) == 0:
        return None
    
    # Get the first crossing of start_db and first crossing of end_db
    time_start = time_array[idx_start[0]]
    time_end = time_array[idx_end[0]]
    
    decay_time = time_end - time_start
    return decay_time, time_start, time_end

# Reference time at 0 dB (peak amplitude)
idx_0db = np.where(envelope_db <= 0)[0]
if len(idx_0db) > 0:
    ref_time = time_after_clap[idx_0db[0]]
else:
    ref_time = time_after_clap[0]

# Calculate EDT (Early Decay Time): 0 dB to -10 dB, extrapolate to -60 dB
edt_result = find_decay_time(envelope_db, time_after_clap, fs, 0, -10)
if edt_result:
    edt_raw, edt_start, edt_end_measured = edt_result
    # Extrapolate to 60 dB
    edt = edt_raw * (60 / 10)
    edt_end = ref_time + edt
else:
    edt, edt_start, edt_end = None, None, None

# Calculate T20: 0 dB to -20 dB, then extrapolate to -60 dB
t20_result = find_decay_time(envelope_db, time_after_clap, fs, 0, -20)
if t20_result:
    t20_raw, _, t20_end_measured = t20_result
    # Extrapolate to 60 dB
    t20 = t20_raw * (60 / 20)
    t20_start = ref_time
    t20_end = ref_time + t20
else:
    t20, t20_start, t20_end = None, None, None

# Calculate T30: 0 dB to -30 dB, then extrapolate to -60 dB
t30_result = find_decay_time(envelope_db, time_after_clap, fs, 0, -30)
if t30_result:
    t30_raw, _, t30_end_measured = t30_result
    # Extrapolate to 60 dB
    t30 = t30_raw * (60 / 30)
    t30_start = ref_time
    t30_end = ref_time + t30
else:
    t30, t30_start, t30_end = None, None, None

# Calculate RT60: 0 dB to -60 dB
rt60_result = find_decay_time(envelope_db, time_after_clap, fs, 0, -60)
if rt60_result:
    rt60, _, rt60_end = rt60_result
    rt60_start = ref_time
else:
    rt60, rt60_start, rt60_end = None, None, None

# Print results
print("\n=== Reverberation Parameters (all extrapolated to 60 dB) ===")
print(f"EDT (extrapolated from 0 to -10 dB):     {edt:.4f} s" if edt else "EDT: Could not calculate")
print(f"T20 (extrapolated from 0 to -20 dB):     {t20:.4f} s" if t20 else "T20: Could not calculate")
print(f"T30 (extrapolated from 0 to -30 dB):     {t30:.4f} s" if t30 else "T30: Could not calculate")
print(f"RT60 (measured from 0 to -60 dB):        {rt60:.4f} s" if rt60 else "RT60: Could not calculate")

In [ ]:
# Plot the raw audio signal with the clap/pulse start time marked

fig_raw, ax_raw = plt.subplots(figsize=(8, 5))

# Convert audio indices to time
time_audio = np.arange(len(audio)) / fs

# Plot raw audio signal
ax_raw.plot(time_audio, audio, 'b-', linewidth=0.5, label='Raw Audio Signal')

# Mark the clap detection point
#ax_raw.plot(clap_time, audio[clap_idx], 'ro', markersize=10, label=f'Peak Amplitude')

# Labels and formatting
ax_raw.set_xlabel('Time (s)', fontsize=12, fontweight='bold')
ax_raw.set_ylabel('Amplitude', fontsize=12, fontweight='bold')
ax_raw.set_title('Raw Audio Signal with Detected Pulse Start', fontsize=14, fontweight='bold')
ax_raw.grid(True, alpha=0.3)
ax_raw.set_ylim(audio.min(), audio.max())
ax_raw.axvline(clap_time, color='r', linestyle='--', linewidth=2, label=f'Clap Detected at {clap_time:.4f} s')
ax_raw.legend(loc='upper right', fontsize=10)

plt.tight_layout()
plt.show()

print(f"Raw audio plotted. Pulse peak detected at {clap_time:.4f} seconds.")

In [ ]:
# Step 4: Plot the envelope and mark the reverberation times

fig, ax = plt.subplots(figsize=(8, 5))

# Plot the decay envelope
ax.plot(time_after_clap, envelope_db, 'b-', linewidth=2, label='Decay Envelope')

# Color map for different metrics
colors = {'EDT': 'green', 'T20': 'red', 'T30': 'orange', 'RT60': 'purple'}

# Plot EDT markers (line goes from 0 to -10 dB, but time is extrapolated to 60 dB)
if edt and edt_start is not None and edt_end_measured is not None:
    ax.axvline(edt_start, color=colors['EDT'], linestyle='--', linewidth=1.5, alpha=0.7)
    ax.axvline(edt_end_measured, color=colors['EDT'], linestyle='--', linewidth=1.5, alpha=0.7)
    ax.plot([edt_start, edt_end_measured], [0, -10], 'o-', color=colors['EDT'], markersize=8, label=f'EDT = {edt:.4f} s (ext. to 60dB)')
    # Label at the -10 dB intersection point
    ax.text(edt_end_measured, -10, f'  EDT\n  {edt:.4f}s', ha='left', va='top', fontsize=9, 
            bbox=dict(boxstyle='round', facecolor=colors['EDT'], alpha=0.3))

# Plot T20 markers (line goes from 0 to -20 dB, but time is extrapolated to 60 dB)
if t20 and t20_start is not None and t20_end_measured is not None:
    ax.axvline(t20_start, color=colors['T20'], linestyle=':', linewidth=1.5, alpha=0.7)
    ax.axvline(t20_end_measured, color=colors['T20'], linestyle=':', linewidth=1.5, alpha=0.7)
    ax.plot([t20_start, t20_end_measured], [0, -20], 's-', color=colors['T20'], markersize=8, label=f'T20 = {t20:.4f} s (ext. to 60dB)')
    # Label at the -20 dB intersection point
    ax.text(t20_end_measured, -20, f'  T20\n  {t20:.4f}s', ha='left', va='top', fontsize=9,
            bbox=dict(boxstyle='round', facecolor=colors['T20'], alpha=0.3))

# Plot T30 markers (line goes from 0 to -30 dB, but time is extrapolated to 60 dB)
if t30 and t30_start is not None and t30_end_measured is not None:
    ax.axvline(t30_start, color=colors['T30'], linestyle=':', linewidth=1.5, alpha=0.7)
    ax.axvline(t30_end_measured, color=colors['T30'], linestyle=':', linewidth=1.5, alpha=0.7)
    ax.plot([t30_start, t30_end_measured], [0, -30], '^-', color=colors['T30'], markersize=8, label=f'T30 = {t30:.4f} s (ext. to 60dB)')
    # Label at the -30 dB intersection point
    ax.text(t30_end_measured, -30, f'  T30\n  {t30:.4f}s', ha='left', va='top', fontsize=9,
            bbox=dict(boxstyle='round', facecolor=colors['T30'], alpha=0.3))

# Plot RT60 markers (line goes from 0 to -60 dB)
if rt60 and rt60_start is not None and rt60_end is not None:
    ax.axvline(rt60_start, color=colors['RT60'], linestyle='-.', linewidth=1.5, alpha=0.7)
    ax.axvline(rt60_end, color=colors['RT60'], linestyle='-.', linewidth=1.5, alpha=0.7)
    ax.plot([rt60_start, rt60_end], [0, -60], 'd-', color=colors['RT60'], markersize=8, label=f'RT60 = {rt60:.4f} s (measured to 60dB)')
    # Label at the -60 dB intersection point
    ax.text(rt60_end, -60, f'  RT60\n  {rt60:.4f}s', ha='left', va='top', fontsize=9,
            bbox=dict(boxstyle='round', facecolor=colors['RT60'], alpha=0.3))

# Add reference lines for key dB levels
ax.axhline(0, color='k', linestyle='-', linewidth=0.5, alpha=0.3)
# ax.axhline(-10, color='gray', linestyle='--', linewidth=0.5, alpha=0.3, label='-10 dB')
# ax.axhline(-20, color='gray', linestyle='--', linewidth=0.5, alpha=0.3, label='-20 dB')
# ax.axhline(-30, color='gray', linestyle='--', linewidth=0.5, alpha=0.3, label='-30 dB')
# ax.axhline(-60, color='gray', linestyle='--', linewidth=0.5, alpha=0.3, label='-60 dB')

# Labels and formatting
ax.set_xlabel('Time (s)', fontsize=12, fontweight='bold')
ax.set_ylabel('Magnitude (dB)', fontsize=12, fontweight='bold')
ax.set_title('Reverberation Time Metrics (extrapolated to 60 dB)', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)
ax.legend(loc='upper right', fontsize=10)
ax.set_xlim(left=0)

plt.tight_layout()
plt.show()

print("\nPlot completed with measured decay lines and extrapolated reverberation times.")